# **Diplomado IA: Aplicaciones 2 - Audio y Video**. <br> Práctico 44: Deep Fakes - Clonación de voz
---
---

**Profesor:**
- Carlos Aspillaga


Nota: Algunas partes de este laboratorio se construyeron usando recursos de las siguientes fuentes:
* https://colab.research.google.com/github/tugstugi/dl-colab-notebooks/blob/master/notebooks/RealTimeVoiceCloning.ipynb
* https://towardsdatascience.com/how-to-produce-a-deepfake-video-in-5-minutes-513984fd24b6
* https://colab.research.google.com/github/Tyler-Hilbert/AudioProcessingInPythonWorkshop/blob/master/AudioProcessingInPython.ipynb#scrollTo=AEvn0yZKNCV4

# **1. Clonación de Voz**

Para clonar la voz de Obama, usaremos un modelo preentrenado

❗❗❗**Nota importante**: ❗❗❗

Ir a "Entorno de Ejecución" > "Cambiar entorno de ejecución"  y seleccionar la versión 2025.07 de Google Colab

### Descarga de código e instalación de librerías necesarias

In [1]:
!pip3 install -U scipy
!git clone https://github.com/jnordberg/tortoise-tts.git
%cd tortoise-tts
!pip install numba
!pip install llvmlite
!pip install transformers==4.29.2
!pip3 install -r requirements.txt
!pip3 install einops==0.5.0
!pip3 install rotary_embedding_torch==0.1.5 unidecode==1.3.5
!python3 setup.py install
!pip install audio2numpy
!apt-get install -qq libportaudio2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 78.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plotnine 0.14.6 requires scipy<1.16.0,>=1.8.0, but you have scipy 1.17.1 which is incompatible.
Cloning into 'tortoise-tts'...
remote: Enumerating objects: 1481, done.
remote: Total 1481 (delta 0), reused 0 (delta 0), pack-reused 1481 (from 1)
Receiving objects: 100% (1481/1481), 53.56 MiB | 42.52 MiB/s, done.
Resolving deltas: 100% (604/604), done.
/content/tortoise-tts
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 64.4 

In [2]:
!pip install "tokenizers==0.13.3" "transformers==4.29.2"

DEPRECATION: Loading egg at /usr/local/lib/python3.11/dist-packages/progressbar-2.5-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.11/dist-packages/TorToiSe-2.3.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [3]:
from transformers import __version__
print(__version__)

4.29.2


In [4]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import IPython
from tortoise.api import TextToSpeech
from tortoise.utils.audio import load_audio, load_voice, load_voices
import sys
from IPython.display import display, Audio, clear_output
from IPython.utils import io
import ipywidgets as widgets
import numpy as np
from pathlib import Path
import soundfile as sf
import audio2numpy as a2n


Como input al modelo preentrenado, necesitamos entregar un audio de referencia, para que el modelo pueda calcular un descriptor de la voz.

Ejecutando las siguientes celdas, usted estará descargando el archivo de Demo provisto por el profesor. Se descargarán los primeros 90 segundos del video https://www.youtube.com/watch?v=sTFWC1PiLVE donde aparece Barack Obama hablando.

## Video de referencia de voz (Demo)

In [5]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/sTFWC1PiLVE?si=ZWuMADnnYIPTkE7h" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [6]:
if not os.path.exists("/content/obama_voice.mp3"):
    os.system('wget -q https://www.dropbox.com/s/bxa9ivwuveu3s3q/obama_voice.mp3 -O /content/obama_voice.mp3')
    os.system('ffmpeg -y -loglevel error -stats -i /content/obama_voice.mp3 -ac 1 -ab 64000 -ar 22050 -t 90 /content/reference_voice.wav')
    audio, sampling_rate = a2n.audio_from_file("/content/reference_voice.wav")
    display(Audio(audio, rate=22050, autoplay=False))

A continuación particionamos el audio en 3 clips de 10 segundos cada uno

In [7]:
!mkdir /content/tortoise-tts/tortoise/voices/obama
!ffmpeg -loglevel error -y -ss 0 -t 10 -i /content/reference_voice.wav /content/tortoise-tts/tortoise/voices/obama/clip_0.wav
!ffmpeg -loglevel error -y -ss 40 -t 10 -i /content/reference_voice.wav /content/tortoise-tts/tortoise/voices/obama/clip_1.wav
!ffmpeg -loglevel error -y -ss 80 -t 10 -i /content/reference_voice.wav /content/tortoise-tts/tortoise/voices/obama/clip_2.wav

A continuación instanciamos el modelo (y se descargan los pesos preentrenados)

In [8]:
tts = TextToSpeech()
voice_samples, conditioning_latents = load_voice('obama')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

...

Done.


100% (25193729 of 25193729) |############| Elapsed Time: 0:00:00 Time:  0:00:00
  0% (0 of 100715777) |                  | Elapsed Time: 0:00:00 ETA:  --:--:--

Done.


100% (100715777 of 100715777) |##########| Elapsed Time: 0:00:01 Time:  0:00:01


Done.


A continuación podemos entregar un texto de referencia y el modelo nos generará la voz (en este caso de Obama) diciendo esa frase.

*Nota: Cada vez que se ejecuta el código se genera un audio diferente.*

In [9]:
text = "This is the last class. I hope you enjoy it."
preset = "ultra_fast" # Options: {"ultra_fast", "fast" (default), "standard", "high_quality"}
gen = tts.tts_with_preset(text, voice_samples=voice_samples, conditioning_latents=conditioning_latents, preset=preset)
torchaudio.save(f'/content/generated.wav', gen.squeeze(0).cpu(), 24000)
IPython.display.Audio(f'/content/generated.wav')

Generating autoregressive samples..


100%|██████████| 1/1 [00:27<00:00, 27.92s/it]


Computing best candidates using CLVP and CVVP


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 1/1 [00:05<00:00,  5.37s/it]


Transforming autoregressive outputs into audio..


100%|██████████| 30/30 [00:01<00:00, 16.92it/s]


## Actividad — comparación de presets

`tts_with_preset` expone cuatro configuraciones. Se comparan `ultra_fast` (el usado en
el ejemplo de clase) y `fast` (el preset por defecto de la librería), midiendo tiempo de
cómputo y duración del audio resultante.


In [10]:
import time, torch, torchaudio
from IPython.display import Audio, display

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU")
print()

resultados = []
for p in ["ultra_fast", "fast"]:
    t0 = time.time()
    g = tts.tts_with_preset(text, voice_samples=voice_samples,
                            conditioning_latents=conditioning_latents, preset=p)
    dt = time.time() - t0
    torchaudio.save(f"/content/cmp_{p}.wav", g.squeeze(0).cpu(), 24000)
    dur = g.shape[-1] / 24000
    resultados.append((p, dt, dur))
    print(f"{p:>11} | cómputo: {dt:6.1f} s | audio: {dur:5.2f} s | RTF: {dt/dur:5.2f}x")
    display(Audio(f"/content/cmp_{p}.wav"))

print()
print(f"fast costó {resultados[1][1]/resultados[0][1]:.1f}x el cómputo de ultra_fast")


GPU: Tesla T4

Generating autoregressive samples..


100%|██████████| 1/1 [00:36<00:00, 36.35s/it]


Computing best candidates using CLVP and CVVP


100%|██████████| 1/1 [00:01<00:00,  1.47s/it]


Transforming autoregressive outputs into audio..


...

100%|██████████| 6/6 [00:09<00:00,  1.60s/it]


Transforming autoregressive outputs into audio..


100%|██████████| 80/80 [00:09<00:00,  8.44it/s]


       fast | cómputo:  203.2 s | audio:  4.22 s | RTF: 48.10x



fast costó 4.1x el cómputo de ultra_fast


### Observaciones

Medido en Tesla T4:

| Preset | Cómputo | Audio | RTF | AR | CLVP+CVVP | Difusión |
|---|---:|---:|---:|---:|---:|---:|
| `ultra_fast` | 50,1 s | 3,81 s | 13,2× | 36,4 s (73 %) | 1,5 s (3 %) | 1,7 s (3 %) |
| `fast` | 203,2 s | 4,22 s | 48,1× | 178,2 s (88 %) | 9,6 s (5 %) | 9,5 s (5 %) |

1. **El costo lo domina el decodificador autoregresivo (73–88 %), no el modelo de
   difusión (3–5 %).** El difusor opera sobre espectrogramas MEL, ~256× comprimidos
   respecto de la onda; el autoregresivo decodifica token a token, sin paralelizar.

2. **La difusión se ralentiza 2,14× por paso entre ambos presets** (18,05 → 8,44 it/s).
   Corresponde al parámetro `cond_free`, que `ultra_fast` desactiva: la difusión sin
   condicionamiento hace dos pasadas por paso y las combina como
   `salida = cond·(k+1) − uncond·k` con k=2, es decir *classifier-free guidance* con
   escala 3 — el mismo mecanismo que el `guidance_scale` de Stable Diffusion. La
   documentación del autor la describe como lo que *"mejora dramáticamente el realismo"*.
   Es la causa principal del timbre plano de `ultra_fast`, por sobre la reducción de
   pasos de difusión.

3. **El re-ranking cuesta 92 ms por candidato (2,9 % del total).** CLVP puntúa sobre
   tokens discretos, sin invocar al difusor; difundir los 16 candidatos habría costado
   26,6 s en lugar de 1,47 s. Es una decisión de diseño explícita del paper (§2.3).

4. **La duración del audio varía 10,8 % entre corridas con la misma frase** (3,81 vs
   4,22 s), por el muestreo estocástico del autoregresivo (`top_p=0.8`, `temp=0.8`).
   La sincronización con el video debe hacerse sobre la duración medida, no estimada.

Se adopta `preset="fast"` con `k=3`: los 96 candidatos se generan de todas formas, por
lo que devolver 3 solo agrega dos pasadas de difusión (+11 % de cómputo) frente al
+200 % de re-ejecutar la celda tres veces.


In [11]:
## Actividad — generación con frase y parámetros propios


In [12]:
import time, torchaudio
from IPython.display import Audio, display

mi_texto = "Congratulations, class of twenty twenty-six. The future is yours to build."

t0 = time.time()
gen = tts.tts_with_preset(
    mi_texto,
    voice_samples=voice_samples,
    conditioning_latents=conditioning_latents,
    preset="fast",
    k=3,                    # 3 candidatos por +11% de cómputo
    clvp_cvvp_slider=0.35,  # <0.5 -> prioriza parecido de voz sobre fidelidad al texto
)
print(f"cómputo: {time.time()-t0:.1f} s")

cands = gen if isinstance(gen, list) else [gen]
for i, w in enumerate(cands):
    p = f"/content/generated_{i}.wav"
    torchaudio.save(p, w.squeeze(0).cpu(), 24000)
    print(f"candidato {i}: {w.shape[-1]/24000:.2f} s")
    display(Audio(p))


Generating autoregressive samples..


100%|██████████| 6/6 [08:32<00:00, 85.33s/it]


Computing best candidates using CLVP and CVVP


100%|██████████| 6/6 [00:09<00:00,  1.61s/it]


Transforming autoregressive outputs into audio..


100%|██████████| 80/80 [00:17<00:00,  4.45it/s]


cómputo: 578.4 s
candidato 0: 6.12 s


candidato 1: 7.05 s


candidato 2: 6.87 s
